# Text Classification

## Loading the file containing all the tracks. Importing tracks

In this step, we import the artists, their tracks, and the lemmatized track data generated in the previous notebook prior to topic modeling.

In [1]:

import json

with open('dataset/tracks_lyrics_preprocessed.json', 'r', encoding="utf-8") as json_file:
    artists = json.load(json_file)
    
print(artists[0]["artist_id"], "\n", 
      artists[0]["artist_name"], "\n", 
      artists[0]['genres'], "\n",
      artists[0]['tracks'][0]['title'], "\n",
      artists[0]['tracks'][0]['topic_word'], "\n",
      artists[0]['tracks'][0]['topic_confidence_%'], "\n", 
      artists[0]['tracks'][0]['lyrics'][:100].lstrip(), "... \n", 
      artists[0]['tracks'][0]['lemmatized_lyrics'][:100].lstrip(), "...", 
      sep="")

582KhTHEVOONNQLmQ5612r
Calcutta 
['italian', 'indie', 'pop', 'singer-songwriter', 'italiano']
Tutti
ballare
32.57
Ho messo le scarpe nuove per i giorni di fango
Forse i leghisti lì in riva al Po non hanno più un ca... 
mettere scarpa nuovo giorno fango forse leghista lì riva po' non avere più capobranco rivoluzione ve...


As we can see, the dataset features over 1,100 unique genres. Note that each artist is associated with 5 of these genres, which were retrieved from Last.fm.

In [2]:
different_genres = set()

for artist in artists:
    for genre in artist['genres']:
        different_genres.add(genre)

print(f"Number of different genres: {len(different_genres)}")

Number of different genres: 1110


## Genres pre-processing

### Rule-Based Taxonomy: Polyhierarchical Genre Mapping

Raw genre metadata extracted from digital platforms is notoriously noisy: it is often a "folksonomy" (an unstructured, user-generated web of highly granular micro-genres). Feeding hundreds of distinct, fragmented tags into an analytical model leads to extreme dimensionality and data sparsity. To resolve this, we implement a **rule-based taxonomy system** to aggregate these micro-genres into a standardized, mathematically manageable set of macro-categories.

Here is a methodological breakdown of our classification pipeline:

* **`PARENT_GENRE_RULES` (The Heuristic Dictionary):** We define a structured dictionary where the keys represent our target macro-genres (e.g., 'Rock', 'Electronic') and the values are lists of substring keywords. By using substring matching (`any(keyword in clean_genre)`), we can catch dozens of variations with a single rule (e.g., the keyword 'metal' automatically captures 'heavy metal', 'nu metal', and 'black metal').
* **The Mapping Logic (`defaultdict`):** Instead of manually initializing empty lists for every parent genre, we utilize Python's `collections.defaultdict(list)`. This data structure is highly efficient for classification tasks, as it automatically creates a new list the first time a key is called, preventing `KeyError` exceptions and keeping the memory footprint minimal.
* **Handling Unmapped Outliers:** It is inevitable that some obscure or overly niche tags will fail to match any of our predefined rules. If a genre is not caught by the 8 macro-categories in `PARENT_GENRE_RULES`, it is not forced into an arbitrary bucket. Instead, it is cleanly isolated into an `unmapped_genres` array and effectively discarded from the main taxonomy. This strict filtering mechanism guarantees that our final dataset remains semantically pure and focused entirely on our analytical scope.

#### Architectural Note: The Polyhierarchical Approach
It is crucial to highlight a specific design choice inside the `for` loop: **we intentionally omitted the `break` statement** after a keyword match is found. 

Standard classification assumes mutual exclusivity (a track is either 'Rock' OR 'Electronic'). However, music is inherently hybrid. A micro-genre like *"indie pop"* conceptually belongs to both the 'Indie' and 'Pop' macro-categories. By allowing the iteration to continue through all rules even after finding a match, a single child genre can be dynamically assigned to multiple parent categories. This creates a **polyhierarchical taxonomy** that accurately reflects the nuanced, overlapping reality of musical genres, preventing arbitrary data loss. 

Furthermore, during the final output generation, we use `list(dict.fromkeys(child_genres))` to remove duplicate entries. This specific Python idiom is a best practice: unlike converting the list to a `set()` (which destroys the order of elements), using dictionary keys inherently deduplicates the data while preserving the original deterministic sorting.

In [3]:
from collections import defaultdict

# Define the taxonomy rules (Keywords to match against)
PARENT_GENRE_RULES = {
    'Hip-Hop': ['hip hop', 'hip-hop', 'rap', 'trap', 'drill', 'grime', 'rnb', 'urban', 'hiphop'],
    'Classical': ['classical', 'opera', 'baroque', 'baritone', 'soprano', 'tenor', 'choir', 'orchestral', 'piano', 'violin', 'chamber', 'renaissance', 'early music'],
    'Pop': ['pop', 'eurodance', 'dance pop', 'electropop', 'teen pop', 'itpop', 'synthpop', 'hyperpop'],
    'Indie': ['indie', 'alternative'],
    'Rock': ['rock', 'punk', 'metal', 'hard rock', 'progressive', 'grunge', 'post-punk', 'hardcore'],
    'Jazz': ['jazz', 'swing', 'bossa nova', 'blues', 'fusion', 'soul'],
    'Electronic': ['electronic', 'techno', 'house', 'dance', 'ambient', 'chillout', 'trance', 'downtempo', 'idm', 'lo-fi', 'dubstep', 'drum and bass', 'electro', 'synthwave', 'club', 'edm'],
    'Cantautore': ['cantautore', 'cantautori', 'singer-songwriter', 'folk', 'canzone d autore']
}

def map_genres_to_multiple_parents(genres_list):
    # Dictionary to store the final mapping: { Parent: [Child1, Child2, ...] }
    mapped_taxonomy = defaultdict(list)
    unmapped_genres = []

    for genre in genres_list:
        clean_genre = genre.strip().lower()
            
        is_mapped = False
        
        # Check for keyword matches in all taxonomy rules (No break statement)
        for parent, keywords in PARENT_GENRE_RULES.items():
            if any(keyword in clean_genre for keyword in keywords):
                # Append the original formatted genre name to the parent group
                mapped_taxonomy[parent].append(genre)
                is_mapped = True
                # REMOVED 'break' here so a child can belong to multiple parents
                
        if not is_mapped:
            unmapped_genres.append(genre)
            
    return mapped_taxonomy, unmapped_genres

# Execute the mapping function
mapped_results, leftovers = map_genres_to_multiple_parents(different_genres)

# Print the organized taxonomy
print("### MULTI-PARENT MAPPED GENRES:")
for parent_genre, child_genres in sorted(mapped_results.items()):
    print(f"\n## {parent_genre}")
    # Remove duplicates from the sublist while preserving order
    unique_children = list(dict.fromkeys(child_genres))
    for child in unique_children:
        print(f"  - {child}")

print("\n" + "-"*40)
print(f"### SKIPPED OR UNMAPPED TAGS ({len(leftovers)}):")
print(", ".join(dict.fromkeys(leftovers)))

### MULTI-PARENT MAPPED GENRES:

## Cantautore
  - combat folk
  - cantautore napoletano
  - Folk Power Metal
  - canzone d autore
  - cantautori
  - Irish Folk
  - salento folklore
  - folk-rock
  - italian  folk
  - folk rock
  - folk
  - italian folk metal
  - Celtic Folk Metal
  - indie folk
  - singer-songwriter
  - italian folk
  - folk metal
  - balfolk
  - Cantautore
  - Andean Folk
  - hip-folk
  - kurdish folk
  - folklore

## Classical
  - classical guitar
  - classical singer
  - pop opera
  - mezzo-soprano
  - early music
  - italian baroque
  - mezzosoprano
  - Italian Opera
  - Classical
  - soprano
  - operatic pop
  - Baroque Italienne XVIIe-XVIIIe Siecle
  - modern classical
  - Romantic Classical
  - violinist
  - chamber music
  - opera
  - chamber pop
  - female opera singer
  - baroque
  - neoclassical fantasy music
  - countertenor
  - baroque music
  - neoclassical
  - orchestral
  - piano
  - jazz piano
  - piano pop
  - classical guitarist
  - baritone
  - ren

Based on the five micro-genres assigned to each artist, we have mapped them to one or more broader parent (macro) genres.

In [4]:
def assign_macro_genres(tags_artista):

    macro_genres_assigned = set()
    
    if not isinstance(tags_artista, list):
        return []
        
    for tag in tags_artista:
        clean_tag = tag.strip().lower()
            
        for parent, keywords in PARENT_GENRE_RULES.items():
            if any(keyword in clean_tag for keyword in keywords):
                macro_genres_assigned.add(parent)
                
    return list(macro_genres_assigned)

for artist in artists:
    artist['macro_genres'] = assign_macro_genres(artist['genres'])
    
    if not artist['macro_genres']:
        artist['macro_genres'] = None

However, some artists have no associated genres, so we need to remove them from the dataset.

In [5]:
import pandas as pd

df = pd.DataFrame(artists).dropna(subset=['macro_genres'])

print("Number of artists with assigned macro-genres:", len(df))

Number of artists with assigned macro-genres: 1286


## Machine Learning

### Multi-Label Genre Classification: Support Vector Machines and Grid Search

Our objective is to train a predictive algorithm capable of deducing the musical macro-genres of an artist based solely on its lyrical content. 

Because music is inherently hybrid and a track can belong to multiple genres simultaneously (e.g., 'Indie' and 'Pop'), this is not a standard multi-class problem, but a **multi-label classification** task. To achieve optimal predictive performance, we deploy a robust Support Vector Machine (SVM) pipeline strictly optimized through exhaustive parallel computing.

* **Target Binarization (`MultiLabelBinarizer`):** Scikit-learn algorithms cannot natively process lists of text strings as target variables. This tool mathematically transforms our lists of genres into a binary matrix. If we have 8 total macro-genres, it creates an array of 8 bits for each track (e.g., `[1, 0, 0, 1, 0, 0, 0, 0]`), where `1` indicates the presence of a specific genre.
* **The Pipeline Architecture:** We encapsulate our workflow into a `Pipeline`. This ensures that the raw text is first vectorized into a mathematical space (`TfidfVectorizer`) and immediately passed to the algorithmic classifier without breaking the data flow.
* **The Classifier (`OneVsRestClassifier` + `LinearSVC`):** Support Vector Machines are exceptionally powerful for text classification because they handle high-dimensional, sparse data (like our TF-IDF matrices) flawlessly. However, standard SVMs are binary classifiers. We solve this by wrapping `LinearSVC` in a `OneVsRestClassifier` (OvR) strategy. Functionally, this trains 8 separate, independent models behind the scenes, one for each genre.
* **Hyperparameter Tuning (`GridSearchCV`):** An algorithm's default settings are rarely optimal. We construct a multi-dimensional grid of hyperparameter combinations that testing different TF-IDF configurations (like incorporating bigrams to capture context such as "hip hop") alongside the SVM's internal geometry. Specifically, we tune the $C$ parameter (the regularization penalty), mapping the boundary between an underfitted generalized model and an overfitted, rigid one.

#### Analytical Note: Imbalanced Data and Macro F1-Scoring
A critical design choice in this code is the scoring metric used by the Grid Search: `scoring='f1_macro'`. Real-world musical datasets are notoriously imbalanced; you will always have significantly more 'Pop' tracks than 'Classical' or 'Jazz' tracks. 

If we optimized for raw "accuracy," the algorithm would cheat: it would learn to simply guess the majority class every time, achieving high accuracy while completely ignoring minority genres. The Macro F1-Score prevents this. It calculates the F1-Score (the harmonic mean of Precision and Recall) independently for every single genre, and then averages them together *unweighted*. This brutally forces the algorithm to treat predicting a rare genre as equally important as predicting a common one. Furthermore, injecting `class_weight='balanced'` into the `LinearSVC` mathematically penalizes the algorithm more heavily if it makes mistakes on minority classes.

**Bibliographic reference (SVM for Text Categorization):**
> Joachims, T. (1998). **Text categorization with support vector machines: Learning with many relevant features**. In *European conference on machine learning* (pp. 137-142). Springer, Berlin, Heidelberg.

In [6]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

# tracks is a list of dictionaries, we need to concatenate the lemmatized lyrics of all tracks into a single string for each artist
X = df['tracks'].apply(lambda tracks: " ".join(track['lemmatized_lyrics'] for track in tracks))

# Transform the list of macro-genres into a binary matrix for multilabel classification
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['macro_genres'])

# Split the dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# DEFINE THE BASE PIPELINE
base_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', OneVsRestClassifier(LinearSVC(class_weight='balanced', random_state=42, max_iter=12000)))
])

# DEFINE THE PARAMETER GRID
param_grid = {
    # TF-IDF: Vocabulary Size & Context
    'tfidf__max_features': [10000, 20000, 40000],
    'tfidf__ngram_range': [(1, 1), (1, 2)], # Test unigrams, bigrams
    
    # TF-IDF: Document Frequency Filters
    'tfidf__min_df': [10], # Keep only words that appear in at least 10 documents (artists)
    'tfidf__max_df': [0.70], # Ignore extremely common stop-words
    'tfidf__use_idf': [True, False], # Test if weighting rare words actually helps
    
    # LinearSVC: Algorithm Tuning
    'clf__estimator__C': [0.01, 0.1, 1.0, 10.0, 100.0], # Penalty parameter (Low = generalized, High = overfitted)
    'clf__estimator__loss': ['hinge', 'squared_hinge'], # The mathematical formula used to calculate errors
}

# Initialize Grid Search with all CPU cores for maximum speed
grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    cv=3, # 3-fold cross validation
    scoring='f1_macro', # Focuses on all genres equally
    n_jobs=-1, # CRITICAL: -1 uses ALL available CPU cores 100%
    verbose=0 # no printing
)

# Run the grid search
print("Initiating Grid Search...")
grid_search.fit(X_train, y_train)

# Print the best hyperparameters and the corresponding macro F1-score
print("\n" + "="*50)
print(f"Best Hyperparameters:\n{grid_search.best_params_}")
print(f"Best Macro F1-Score: {grid_search.best_score_:.4f}")
print("="*50 + "\n")

# Evaluate
best_model_SVM = grid_search.best_estimator_
y_pred = best_model_SVM.predict(X_test)

print("\n### CLASSIFICATION REPORT (OneVsRestClassifier LinearSVC) ###")
svm_report = classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0)
svm_report_dict = classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0, output_dict=True)
print(svm_report)

Initiating Grid Search...

Best Hyperparameters:
{'clf__estimator__C': 0.1, 'clf__estimator__loss': 'squared_hinge', 'tfidf__max_df': 0.7, 'tfidf__max_features': 10000, 'tfidf__min_df': 10, 'tfidf__ngram_range': (1, 1), 'tfidf__use_idf': True}
Best Macro F1-Score: 0.4623


### CLASSIFICATION REPORT (OneVsRestClassifier LinearSVC) ###
              precision    recall  f1-score   support

  Cantautore       0.35      0.75      0.48        36
   Classical       0.30      0.90      0.45        30
  Electronic       0.36      0.52      0.42        56
     Hip-Hop       0.66      0.83      0.74        64
       Indie       0.49      0.61      0.54        36
        Jazz       0.24      0.67      0.36        30
         Pop       0.58      0.67      0.62        76
        Rock       0.46      0.42      0.44        73

   micro avg       0.43      0.65      0.51       401
   macro avg       0.43      0.67      0.51       401
weighted avg       0.46      0.65      0.53       401
 samples avg  

### Classifier chain 

To overcome the limitations of isolated predictions and capture the inherent correlations between musical genres, we upgraded the classification architecture from an independent One-vs-Rest (OvR) approach to a **Classifier Chain**.

* **Sequential Dependency Modeling (`ClassifierChain`):** Instead of treating each genre in a mathematical vacuum, this architecture arranges the `LinearSVC` models sequentially. The first classifier predicts the presence of the first genre based solely on the TF-IDF text features. Crucially, the second classifier predicts the second genre utilizing both the original text features *and* the binary prediction of the preceding model. This chain continues until the final genre, allowing the algorithm to learn real-world musical logic (e.g., if an artist is successfully flagged as 'Hip-Hop' early in the chain, the network leverages this data to mathematically lower the probability of classifying it as 'Classical' downstream).

In [7]:
from sklearn.multioutput import ClassifierChain

base_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', ClassifierChain(LinearSVC(class_weight='balanced', random_state=42, max_iter=50000)))
])

grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    cv=3,                 
    scoring='f1_macro',   
    n_jobs=-1, # Uses all CPU cores
    verbose=0
)

print("Starting Grid Search with Classifier Chain...")
grid_search.fit(X_train, y_train)

print("\n" + "="*50)
print(f"Best Hyperparameters Found:\n{grid_search.best_params_}")
print(f"Best Macro F1-Score: {grid_search.best_score_:.4f}")
print("="*50 + "\n")

best_model_Chain = grid_search.best_estimator_
y_pred = best_model_Chain.predict(X_test)

print("\n### CLASSIFICATION REPORT (Classifier Chain LinearSVC) ###")
chain_report = classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0)
chain_report_dict = classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0, output_dict=True)
print(chain_report)

Starting Grid Search with Classifier Chain...

Best Hyperparameters Found:
{'clf__estimator__C': 1.0, 'clf__estimator__loss': 'squared_hinge', 'tfidf__max_df': 0.7, 'tfidf__max_features': 10000, 'tfidf__min_df': 10, 'tfidf__ngram_range': (1, 1), 'tfidf__use_idf': True}
Best Macro F1-Score: 0.4182


### CLASSIFICATION REPORT (Classifier Chain LinearSVC) ###
              precision    recall  f1-score   support

  Cantautore       0.39      0.42      0.41        36
   Classical       0.29      0.77      0.42        30
  Electronic       0.31      0.07      0.12        56
     Hip-Hop       0.65      0.81      0.72        64
       Indie       0.73      0.53      0.61        36
        Jazz       0.40      0.13      0.20        30
         Pop       0.61      0.63      0.62        76
        Rock       0.44      0.32      0.37        73

   micro avg       0.50      0.47      0.48       401
   macro avg       0.48      0.46      0.43       401
weighted avg       0.49      0.47      0.45  

### Extreme Gradient Boosting (XGBoost)

While Support Vector Machines are exceptionally efficient at drawing linear boundaries in high-dimensional sparse spaces (like TF-IDF matrices), they may struggle to capture highly complex, non-linear interactions between words. To explore non-linear decision boundaries, we introduce an advanced ensemble learning architecture: **Extreme Gradient Boosting (XGBoost)**.

We maintain the **Classifier Chain** architecture to preserve the sequential dependencies between musical genres, but replace the internal linear solver with an XGBoost classifier.

* **The Core Estimator (Gradient Boosting):** Unlike a single decision tree, XGBoost builds an ensemble of weak learners (shallow trees) sequentially. Each new tree is specifically trained to correct the residual errors made by the combination of all previous trees. For large-scale text classification, we utilize the `tree_method='hist'` parameter. This enables a histogram-based algorithm that bins continuous features, dramatically reducing computational time and memory consumption without sacrificing predictive accuracy.
* **Algorithmic Synergy with Text Data:** Text data is inherently sparse. XGBoost natively handles sparsity by learning optimal default directions for missing or zero-values at every tree split, making it highly compatible with our TF-IDF vectorization strategy.
* **Hyperparameter Tuning Strategy:** The tuning geometry for a Boosting model is fundamentally different from an SVM. Our `GridSearchCV` focuses on the delicate balance between learning capacity and regularization to prevent overfitting:
  * **Ensemble Size & Step Size (`n_estimators` & `learning_rate`):** These two parameters are strictly correlated. `n_estimators` dictates the total number of sequential trees built. A higher number allows the model to learn more complex patterns but risks overfitting the training data. The `learning_rate` (often denoted as $\eta$) shrinks the feature weights after every boosting step. We test various combinations because a robust model typically requires higher `n_estimators` paired with a lower `learning_rate`.
  * **Tree Complexity (`max_depth`):** This defines the maximum number of nodes from the root to the farthest leaf. In text classification, shallow trees (e.g., depth 3) might fail to capture multi-word context, while overly deep trees (e.g., depth 9+) might memorize specific sentence structures.
* **Resource Management (`n_jobs` threading):** A critical architectural note for this pipeline is the management of parallel computing. Because XGBoost is highly optimized and manages parallel processing internally at the tree-construction level (`n_jobs=-1` inside the classifier), we explicitly restrict the Grid Search to a single thread (`n_jobs=1`). This prevents nested multiprocessing, which would otherwise lead to severe thread contention and memory bottlenecks.

As with previous iterations, the model is strictly evaluated using the **Macro F1-Score** (`f1_macro`) to ensure that the algorithm performs equally well on minority classes (e.g., Jazz, Classical) as it does on dominant ones (e.g., Pop, Hip-Hop).

In [8]:
import xgboost as xgb

base_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    
    # We wrap XGBClassifier inside the ClassifierChain.
    # n_jobs=-1 forces XGBoost to use all available CPU cores.
    # tree_method='hist' is a highly optimized algorithm for large datasets (much faster).
    ('clf', ClassifierChain(xgb.XGBClassifier(
        random_state=42, 
        n_jobs=-1,
        tree_method='hist' 
    )))
])

# XGBoost has different parameters compared to SVM.
param_grid = {
    # 1. TF-IDF Parameters
    'tfidf__max_features': [10000],
    'tfidf__ngram_range': [(1, 1)],
    
    # 2. XGBoost: The Number of Trees
    # More trees = better learning, but takes longer and risks overfitting.
    'clf__estimator__n_estimators': [300],
    
    # 3. XGBoost: Tree Depth
    # How many levels of "if-then" questions a single tree can ask.
    # Standard is 6. Text classification often benefits from slightly deeper trees.
    'clf__estimator__max_depth': [3],
    
    # 4. XGBoost: Learning Rate
    # How much each tree contributes to the final answer. 
    # Usually, if you increase n_estimators, you should lower the learning_rate.
    'clf__estimator__learning_rate': [0.3],
}

grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    # Set to 1 because XGBoost is already managing the multiprocessing internally.
    n_jobs=1, 
    verbose=0
)

print("Starting XGBoost Grid Search...")
grid_search.fit(X_train, y_train)

print("\n" + "="*50)
print(f"Best XGBoost Hyperparameters Found:\n{grid_search.best_params_}")
print(f"Best Macro F1-Score: {grid_search.best_score_:.4f}")
print("="*50 + "\n")

best_model_XGBoost = grid_search.best_estimator_
y_pred = best_model_XGBoost.predict(X_test)

print("\n### CLASSIFICATION REPORT (XGBoost Chain) ###")
xgboost_report = classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0)
xgboost_report_dict = classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0, output_dict=True)
print(xgboost_report)

Starting XGBoost Grid Search...

Best XGBoost Hyperparameters Found:
{'clf__estimator__learning_rate': 0.3, 'clf__estimator__max_depth': 3, 'clf__estimator__n_estimators': 300, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}
Best Macro F1-Score: 0.3370


### CLASSIFICATION REPORT (XGBoost Chain) ###
              precision    recall  f1-score   support

  Cantautore       0.46      0.17      0.24        36
   Classical       0.64      0.23      0.34        30
  Electronic       1.00      0.04      0.07        56
     Hip-Hop       0.78      0.72      0.75        64
       Indie       1.00      0.11      0.20        36
        Jazz       0.50      0.03      0.06        30
         Pop       0.61      0.58      0.59        76
        Rock       0.31      0.42      0.36        73

   micro avg       0.54      0.35      0.42       401
   macro avg       0.66      0.29      0.33       401
weighted avg       0.65      0.35      0.38       401
 samples avg       0.50      0.39    

### Deep Learning for Multi-Label Genre Classification: Implementing UmBERTo

#### The Model: `Musixmatch/umberto-commoncrawl-cased-v1`
To understand the complex semantics of Italian lyrics, we utilize **UmBERTo**, a powerful language model developed by Musixmatch. 
* **Architecture:** It is based on the **RoBERTa** architecture (an optimized version of Google's BERT).
* **Training Data:** It was pre-trained on a massive dataset of Italian web pages (the Italian section of the Common Crawl corpus). 
* **Why this model?** Because it was specifically designed to grasp the grammatical rules, slang, and contextual nuances of the Italian language. Unlike traditional algorithms that look at word frequencies (TF-IDF), UmBERTo reads the lyrics contextually, understanding how words relate to each other within a sentence.

---

#### Pipeline Breakdown: Advanced Architectural Choices

To maximize performance and prevent the model from simply memorizing the dataset (Overfitting), we implemented several advanced Deep Learning techniques. Here is a step-by-step explanation of the pipeline:

**1. Dataset Flattening & Reproducibility**
* **From Artists to Tracks:** We exploded the hierarchical dataset. Instead of feeding the model a massive string of 20 combined songs per artist, we flattened the structure so that **1 row = 1 track**. This prevents catastrophic data loss, dramatically increases the training samples, and provides the network with a much richer vocabulary.
* **Seed Locking:** We strictly froze the random number generators across Python, NumPy, and PyTorch (both CPU and GPU) to ensure all training results are 100% deterministic and reproducible.

**2. Smart Tokenization (Head+Tail Truncation)**
* Transformers have a strict hardware limit of **512 tokens** per input. Instead of blindly truncating songs that exceed this limit, we implemented a semantic **Head+Tail Truncation** strategy. 
* The script extracts the first 150 words (intro/first verse) and the last 150 words (outro/final chorus) of a track, dynamically skipping the repetitive middle section. This ensures the model receives the most contextually rich parts of the song without crashing.

**3. Anti-Overfitting Defenses (Layer Freezing & Early Stopping)**
* **Layer Freezing:** UmBERTo possesses a deep understanding of Italian grammar. To prevent "Catastrophic Forgetting" (where the model destroys its pre-trained linguistic knowledge while trying to learn music genres), we **froze the embeddings and the first 6 encoder layers**. The network is only allowed to calculate gradients and update its top layers, ensuring stable and focused learning.
* **Early Stopping & Weight Decay:** We increased the `weight_decay` to `0.05` for stronger regularization. Furthermore, we integrated an `EarlyStoppingCallback` that continuously monitors the Validation Loss. If the model starts to overfit (memorize) and fails to improve for 2 consecutive epochs, the training halts automatically, saving the most generalized weights.

**4. Addressing the "Class Imbalance" Problem**
* In music datasets, genres like *Pop* or *Hip-Hop* heavily outnumber niche genres like *Jazz*. If left untreated, the neural network would simply learn to always guess the majority classes.
* We subclassed the Hugging Face `Trainer` to inject a custom loss function (`BCEWithLogitsLoss`). We calculate **Positive Weights** (`pos_weights`) mathematically inversely proportional to the frequency of each genre, heavily penalizing the model when it makes errors on minority classes.

**5. Dynamic Threshold Optimization**
* By default, a neural network assumes a rigid probability threshold of 50% (0.5) to assign a label. However, in multi-label classification with imbalanced data, 0.5 is rarely the optimal cutoff.
* Instead of retraining the model multiple times, we extract the raw probability scores (logits converted via a Sigmoid function) and perform a **grid search across different thresholds** (from 40% to 95%).
* The script identifies the exact mathematical threshold that maximizes the Macro F1-Score, applying it to generate the final, optimized `classification_report`.

In [9]:
import pandas as pd
import torch
import numpy as np
from torch import nn
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback)
from sklearn.metrics import f1_score

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

print("Flattening the dataset: 1 Row = 1 Track...")
flat_data = []

# Iterate through the original dataframe row by row
for idx, row in df.iterrows():
    # Extract the artist's genres for this row
    artist_genres = row['macro_genres']
    
    # Create a new independent row for each track by this artist
    for track in row['tracks']:
        flat_data.append({
            'text': track['lemmatized_lyrics'], 
            'macro_genres': artist_genres        # Assign the artist's genres to the track
        })

# Create the new "flattened" DataFrame
df_tracks = pd.DataFrame(flat_data)
print(f"Dataset flattened from {len(df)} artists to {len(df_tracks)} individual tracks!")

# head + tail truncation function
# An average pop song has about 300 words. We want to take the first 150 and the last 150
# discarding the middle part which is often just a repeated chorus.
def get_head_tail(text, max_words=300):
    words = text.split()
    if len(words) > max_words:
        # Take the first 150 and the last 150 words, skipping the center
        return " ".join(words[:150] + words[-150:])
    return text

# Apply the strategic truncation to the text
df_tracks['text_optimized'] = df_tracks['text'].apply(get_head_tail)
X = df_tracks['text_optimized']

# Binarize the labels for multi-label classification
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df_tracks['macro_genres'])

# Filter out texts that are too short
valid_mask = np.array([len(str(text).strip()) > 10 for text in X])
X_clean = X[valid_mask]
y_clean = y[valid_mask]
y = np.array(y_clean, dtype=np.float32)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X_clean.to_numpy(), y, test_size=0.2, random_state=42)

# Model preparatiuon
model_name = "Musixmatch/umberto-commoncrawl-cased-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
num_labels = len(mlb.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    problem_type="multi_label_classification" 
)

# Layer freezing to prevent overfitting
print("\nFreezing the first 6 layers of UmBERTo to prevent Catastrophic Forgetting...")
# UmBERTo has 12 "encoder" layers. We freeze its grammatical foundation (embeddings + first 6 layers)
# so that it only updates the last part of its "brain" to learn musical genres.
for param in model.roberta.embeddings.parameters():
    param.requires_grad = False
for i in range(6):
    for param in model.roberta.encoder.layer[i].parameters():
        param.requires_grad = False

# Hugging Face Dataset preparation
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "labels": y_train.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "labels": y_test.tolist()})

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, # Now it will only truncate if it exceeds the limit after our head+tail cut
        max_length=512
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Weights for class imbalance
positives = y_train.sum(axis=0)
negatives = len(y_train) - positives
pos_weights = negatives / (positives + 1e-5) 

# Custom Trainer
class ImbalancedDatasetTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        device = model.device
        weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=weights_tensor)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs >= 0.3).astype(int)
    f1_macro = f1_score(labels, predictions, average="macro", zero_division=0)
    return {"f1_macro": f1_macro}

# Anti-overfitting Training Arguments
training_args = TrainingArguments(
    output_dir="./risultati_bert_musica",
    eval_strategy="epoch",          
    save_strategy="epoch",
    learning_rate=3e-5,             
    per_device_train_batch_size=8,  
    per_device_eval_batch_size=8,
    num_train_epochs=8,             
    weight_decay=0.05,             
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro", 
    save_total_limit=2              
)

# Trainer with Early Stopping
trainer = ImbalancedDatasetTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    # If Validation Loss / F1 doesn't improve for 2 consecutive epochs, stop the training!
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
)

# Start Training
print("\nStarting Deep Training with Anti-Overfitting Defenses...")
trainer.train()

# Evaluation
print("\nEvaluating on the Test Set...")
predictions = trainer.predict(test_dataset)
probs = 1 / (1 + np.exp(-predictions.predictions))

# Threshold Optimization
print("\nSearching for the perfect threshold...")
thresholds_to_test = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
best_threshold = 0
best_f1 = 0

for threshold in thresholds_to_test:
    y_pred_test = (probs >= threshold).astype(int)
    f1_macro = f1_score(y_test, y_pred_test, average="macro", zero_division=0)
    print(f"Threshold: {threshold:.2f} -> F1-Score Macro: {f1_macro:.4f}")
    if f1_macro > best_f1:
        best_f1 = f1_macro
        best_threshold = threshold

print("\n" + "="*50)
print(f"THE WINNING THRESHOLD IS: {best_threshold:.2f} with an F1 of {best_f1:.4f}")
print("="*50 + "\n")

y_pred_optimized = (probs >= best_threshold).astype(int)
print("### CLASSIFICATION REPORT (OPTIMIZED THRESHOLD) ###")
bert_report = classification_report(y_test, y_pred_optimized, target_names=mlb.classes_, zero_division=0)
bert_report_dict = classification_report(y_test, y_pred_optimized, target_names=mlb.classes_, zero_division=0, output_dict=True)
print(bert_report)

/home/gabriele11231/.venvs/rapids25.06_python3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1781376141.060674   40699 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781376141.193870   40699 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781376142.633564   40699 port.cc:153] oneDNN custom operations are on. You may see slight

Flattening the dataset: 1 Row = 1 Track...
Dataset flattened from 1286 artists to 15135 individual tracks!


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at Musixmatch/umberto-commoncrawl-cased-v1 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Freezing the first 6 layers of UmBERTo to prevent Catastrophic Forgetting...


Map: 100%|██████████| 3015/3015 [00:00<00:00, 11294.39 examples/s]



Starting Deep Training with Anti-Overfitting Defenses...


Epoch,Training Loss,Validation Loss,F1 Macro
1,0.981900,0.944392,0.405401
2,0.925000,0.925937,0.425025
3,0.882300,0.945966,0.430619
4,0.849900,0.955664,0.450430
5,0.800700,0.979204,0.454383
6,0.750400,0.993716,0.450015
7,0.691900,1.049510,0.460395
8,0.668100,1.060243,0.457756



Evaluating on the Test Set...



Searching for the perfect threshold...
Threshold: 0.40 -> F1-Score Macro: 0.4729
Threshold: 0.50 -> F1-Score Macro: 0.4784
Threshold: 0.60 -> F1-Score Macro: 0.4734
Threshold: 0.70 -> F1-Score Macro: 0.4527
Threshold: 0.80 -> F1-Score Macro: 0.3910
Threshold: 0.90 -> F1-Score Macro: 0.2420
Threshold: 0.95 -> F1-Score Macro: 0.0995

THE WINNING THRESHOLD IS: 0.50 with an F1 of 0.4784

### CLASSIFICATION REPORT (OPTIMIZED THRESHOLD) ###
              precision    recall  f1-score   support

  Cantautore       0.38      0.71      0.50       625
   Classical       0.46      0.50      0.48       141
  Electronic       0.22      0.43      0.29       343
     Hip-Hop       0.74      0.79      0.77      1058
       Indie       0.35      0.64      0.45       513
        Jazz       0.16      0.38      0.23       235
         Pop       0.61      0.74      0.67      1165
        Rock       0.33      0.65      0.44       768

   micro avg       0.44      0.68      0.53      4848
   macro avg      

### Classifier Performance Analysis

Before comparing the models, it is essential to understand the metrics used to evaluate their effectiveness. 

#### Evaluation Metrics Definitions
In the context of multi-class classification, "Macro" metrics calculate the score for each class (genre) separately and then compute their unweighted mean. This means every class is treated equally, regardless of its frequency in the dataset.

* **Precision:** Answers the question: *"Out of all the songs the model predicted as belonging to a specific genre, how many actually belonged to it?"*. Mathematically, it is the ratio of True Positives to the sum of True Positives and False Positives. A high Precision indicates that the model produces very few "false alarms" (few false positives).
* **Recall (or Sensitivity):** Answers the question: *"Out of all the songs that actually belonged to a specific genre, how many did the model correctly identify?"*. It is the ratio of True Positives to the sum of True Positives and False Negatives. A high Recall indicates that the model successfully captures most of the correct instances, missing very few (few false negatives).
* **F1-Score:** Since there is often a trade-off between Precision and Recall (improving one usually degrades the other), the F1-Score provides a single unified metric. It is calculated as the *harmonic mean* of Precision and Recall. It is particularly useful when you need to strike a balance between the two metrics or when dealing with imbalanced datasets.

---

#### Global Commentary on the Four Models

Looking at the **Global Performance Comparison (Macro Averages)** at the bottom of the dashboard, distinct behavioral profiles emerge for each approach:

* **XGBoost Chain (Orange): The Conservative.**
    This model achieves the highest **Macro Precision** (0.66) but the lowest **Macro Recall** (under 0.30). This means XGBoost Chain is extremely cautious: it only makes predictions when it is highly confident. Consequently, when it assigns a genre, it is usually right, but it completely misses the majority of the texts, generating a high number of false negatives. This imbalance leads to the lowest overall F1-Score (0.33).
* **BERT (Green): The Eager Predictor.**
    The BERT neural network presents a different profile. It achieves a good **Macro Recall** (0.61, the second highest), but its **Macro Precision** is the lowest among the models (around 0.41). BERT manages to "catch" a large portion of the texts for each genre, but it produces more false positives compared to the other models. This suggests that for this specific lyrical text task, the BERT model might need more fine-tuning to refine its discriminative capabilities.
* **SVM Chain (Yellow): The Middle Ground.**
    The chained Support Vector Machine sits in the middle, offering a decent balance (Precision is 0.48 and Recall is 0.46). However, while it has better precision than BERT and SVM OvR, it is outperformed by both in terms of overall Macro F1-Score.

#### The Best Global Model: SVM OvR

Based on the global macro averages, the winner of this experiment is the **SVM OvR (Support Vector Machine One-vs-Rest)** model.

Looking at the global data, SVM OvR achieves the **Best Macro F1-Score** (reaching 0.50). The reason for its success on a macro level lies in its ability to find cases without sacrificing too much precision:
1.  It achieves the highest **Macro Recall** (0.67), successfully finding the vast majority of texts belonging to the target classes.
2.  At the same time, it maintains a slightly better Precision compared to BERT (0.43 vs 0.41).

#### A Nuanced Look at Per-Genre Performance

While SVM OvR wins on the global *macro* average, analyzing the data **per Musical Genre (Top Chart)** reveals a more complex reality:

* **BERT's Dominance in Standard Genres:** BERT (Green) actually achieves the highest F1-Score in the majority of individual categories: *Cantautore*, *Classical*, *Pop*, *Rock*, and *Hip-Hop*. 
* **SVM OvR's Consistency on Difficult Genres:** The reason SVM OvR wins the overall Macro F1-Score is due to its superior robustness on the hardest genres to classify. SVM OvR stands out with significantly higher performances in *Electronic*, *Indie*, and *Jazz*, whereas BERT and the other models struggle severely (dropping below 0.30 or even 0.10 F1-scores). 

*(Note: Almost all models perform exceptionally well on Hip-Hop, likely due to highly distinct rhythmic text patterns and vocabulary, and struggle on Electronic and Jazz. Ultimately, BERT excels on the more populated/standard genres, but SVM OvR remains the most reliable and versatile option across the entire tested musical spectrum, preventing severe performance drops on the difficult classes.)*

In [10]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Using the dictionary variables you saved during the training phases.
genres = list(mlb.classes_)

# Clean comprehension lists directly querying the dictionaries
df_f1 = pd.DataFrame({
    'Genre': genres,
    'SVM_Grid_F1': [svm_report_dict[g]['f1-score'] for g in genres],
    'SVM_Chain_F1': [chain_report_dict[g]['f1-score'] for g in genres],
    'XGBoost_Chain_F1': [xgboost_report_dict[g]['f1-score'] for g in genres],
    'BERT_F1': [bert_report_dict[g]['f1-score'] for g in genres]
})

def extract_macro(report_dict):
    return [report_dict['macro avg']['precision'], report_dict['macro avg']['recall'], report_dict['macro avg']['f1-score']]

df_macro = pd.DataFrame({
    'Metric': ['Macro Precision', 'Macro Recall', 'Macro F1-Score'],
    'SVM_Grid': extract_macro(svm_report_dict),
    'SVM_Chain': extract_macro(chain_report_dict),
    'XGBoost_Chain': extract_macro(xgboost_report_dict),
    'BERT': extract_macro(bert_report_dict)
})


colors = {
    'SVM_Grid': '#56B4E9',     # Sky Blue
    'SVM_Chain': '#E69F00',    # Orange
    'XGBoost_Chain': '#D55E00',# Vermilion
    'BERT': '#009E73'          # Bluish Green
}


fig = make_subplots(
    rows=2, cols=1, 
    subplot_titles=(
        "F1-Score Comparison per Musical Genre", 
        "Global Performance Comparison (Macro Averages)"
    ),
    vertical_spacing=0.15,
    row_heights=[0.6, 0.4]
)

# CHART 1: GENRES
fig.add_trace(go.Bar(x=df_f1['Genre'], y=df_f1['SVM_Grid_F1'], name='SVM OvR', marker_color=colors['SVM_Grid']), row=1, col=1)
fig.add_trace(go.Bar(x=df_f1['Genre'], y=df_f1['SVM_Chain_F1'], name='SVM Chain', marker_color=colors['SVM_Chain']), row=1, col=1)
fig.add_trace(go.Bar(x=df_f1['Genre'], y=df_f1['XGBoost_Chain_F1'], name='XGBoost Chain', marker_color=colors['XGBoost_Chain']), row=1, col=1)
fig.add_trace(go.Bar(x=df_f1['Genre'], y=df_f1['BERT_F1'], name='BERT', marker_color=colors['BERT']), row=1, col=1)

# CHART 2: MACRO
fig.add_trace(go.Bar(name='SVM OvR', x=df_macro['Metric'], y=df_macro['SVM_Grid'], marker_color=colors['SVM_Grid'], showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(name='SVM Chain', x=df_macro['Metric'], y=df_macro['SVM_Chain'], marker_color=colors['SVM_Chain'], showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(name='XGBoost Chain', x=df_macro['Metric'], y=df_macro['XGBoost_Chain'], marker_color=colors['XGBoost_Chain'], showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(name='BERT', x=df_macro['Metric'], y=df_macro['BERT'], marker_color=colors['BERT'], showlegend=False), row=2, col=1)

fig.update_layout(
    title_text="Classifier Performance Dashboard",
    title_font=dict(size=20, color='black'),
    template='plotly_white',
    barmode='group',
    height=850,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
)

fig.update_yaxes(title_text="F1-Score", range=[0, 1], row=1, col=1)
fig.update_yaxes(title_text="Score (0-1)", range=[0, 1], row=2, col=1)

fig.show()

### Sentiment Analysis for sing with the topic "amore"

In the previous notebook focused on Topic Modeling, we utilized a trained neural network to extract a dominant topic for each track. In this section, we will isolate all the songs classified under the topic of 'Love' (Amore) to analyze the underlying sentiment and explore how the lyrics address this theme.

#### 1. Data Flattening and Preparation
The script begins by transforming a complex, nested data structure into a flat, tabular format using Pandas:
*   **`pd.json_normalize`:** This is a highly efficient Pandas function. Instead of using nested `for` loops to extract tracks from each artist, it "explodes" the `tracks` list so that **1 row = 1 song**. 
*   **Metadata Retention:** By passing `meta=['artist_name']`, the script ensures that every individual track retains the context of its creator, which is crucial for downstream analysis.

#### 2. The Model: `MilaNLProc/feel-it-italian-sentiment`
This is the analytical core of the script. Rather than using a generic multilingual model, the code specifically imports **FEEL-IT**, an advanced Transformer model built for the Italian language.
*   **Architecture:** FEEL-IT is built on top of **UmBERTo** (an Italian RoBERTa model). 
*   **Specialization:** It was explicitly fine-tuned by researchers at the University of Milan (MilaNLProc) to recognize emotion and sentiment in Italian text. Because it was trained on highly expressive and informal texts (like Twitter), it is exceptionally well-suited for analyzing the creative, slang-heavy, and emotional nature of song lyrics.
*   **Hardware Acceleration:** The parameter `device=0` ensures that the Hugging Face pipeline utilizes the GPU (if available), which exponentially speeds up the inference process across thousands of rows.

#### 3. Targeted Filtering ("Amore")
The code isolates the analysis to a single conceptual topic:
*   **Thematic Focus:** By filtering for `TARGET_TOPIC = "amore"`, the script drops irrelevant tracks. This allows you to specifically answer the research question: *"When Italian artists sing about love, is the overall sentiment positive or negative?"*

#### 4. Batch Inference and a Note on Truncation
The code prepares the text and passes it to the model:
*   **The Truncation Nuance:** The code executes `str[:]`, which actually passes the **entire lemmatized string**. This works perfectly because the text has already been lemmatized (stripped of stopwords and noise), making it short enough to naturally fit within the Transformer's strict 512-token memory limit without crashing.
*   **Vectorized Processing:** By passing `df_target['truncated_lyrics'].tolist()` directly into the `sentiment_analyzer`, the Hugging Face pipeline is able to process the data in optimized batches rather than one by one, maximizing computational efficiency.

#### 5. Result Extraction
Finally, the pipeline returns a list of dictionaries containing the predicted labels and confidence scores. These are elegantly unpacked using list comprehensions and assigned back to the DataFrame (`sentiment_label` and `sentiment_score`), resulting in a clean, analyzable dataset ready for visualization.

In [11]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline

# This automatically dives into the 'tracks' list and creates a row for EACH song.
# The 'meta' parameter tells Pandas to carry over the artist info to the song's row.
df_music = pd.json_normalize(
    artists, 
    record_path=['tracks'], 
    meta=['artist_name'])


# Load sentiment model from Hugging Face (FEEL-IT, fine-tuned on Italian text)
print("Initializing the Italian Sentiment Model (FEEL-IT)...")
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="MilaNLProc/feel-it-italian-sentiment", 
    device=0 if torch.cuda.is_available() else -1
)

# Selecting only the tracks that belong to the "amore" topic for sentiment analysis
TARGET_TOPIC = "amore"
print(f"Filtering dataset for tracks assigned to the topic: '{TARGET_TOPIC.upper()}'...")

# Keep only target topic and create a safe copy
df_target = df_music[df_music['topic_word'] == TARGET_TOPIC].copy()

df_target['truncated_lyrics'] = df_target['lemmatized_lyrics'].str[:]

print(f"Found {len(df_target)} tracks. Starting batch inference...")

results = sentiment_analyzer(df_target['truncated_lyrics'].tolist())

df_target['sentiment_label'] = [res['label'] for res in results]
df_target['sentiment_score'] = [np.round(res['score'] * 100, 2) for res in results]

print("\nSample results:")
print("-" * 60)

display_columns = ['title', 'artist_name', 'sentiment_label', 'sentiment_score']
print(df_target[display_columns].head(3).to_string(index=False))

print("-" * 60)
print(f"Successfully analyzed {len(df_target)} tracks.")

Initializing the Italian Sentiment Model (FEEL-IT)...


Device set to use cuda:0


Filtering dataset for tracks assigned to the topic: 'AMORE'...
Found 3195 tracks. Starting batch inference...

Sample results:
------------------------------------------------------------
                title   artist_name sentiment_label  sentiment_score
Cosa mi manchi a fare     Calcutta         negative            99.97
         Paracetamolo     Calcutta         positive            99.97
            Gli occhi Frah Quintale        negative            99.98
------------------------------------------------------------
Successfully analyzed 3195 tracks.


### Sentiment Analysis Results: The Topic of "Love"

The donut chart reveals a striking insight into how Italian artists portray love in their lyrics:

*   **A Melancholic Majority:** A significant **64.5%** of the tracks categorized under the topic of "Love" exhibit a **Negative** sentiment. This suggests that the lyrics in the dataset predominantly focus on heartbreak, longing, unrequited love, or the painful aspects of relationships.
*   **The Joyful Minority:** Only **35.5%** of the songs are classified with a **Positive** sentiment, indicating that purely happy, celebratory, or fulfilled romantic narratives are actually the minority in this context.

**Analytical Conclusion:**
The data clearly demonstrates a strong artistic tendency to use music as an emotional outlet for the complex and often suffering side of love. Rather than celebrating romance, the artists in this dataset are nearly twice as likely to sing about its complications and sorrows. 
It is also important to note that, as discussed during the topic assignment phase, the classification of every track may not be perfectly accurate. This is largely due to the rich semantic nuances, idioms, and poetic complexities inherent in the Italian language.

In [12]:
import plotly.graph_objects as go

# Create a working copy from the target DataFrame
df_viz = df_target.copy()

# Standardize the sentiment labels (capitalize them for a cleaner look)
df_viz['Sentiment'] = df_viz['sentiment_label'].str.capitalize()

# Calculate the total counts for each sentiment
sentiment_counts = df_viz['Sentiment'].value_counts()

# Okabe-Ito Palette: Sky Blue for Positive, Orange for Negative
custom_palette = {'Positive': '#56B4E9', 'Negative': '#E69F00'}

# Map the colors strictly to the available labels to prevent mismatches
pie_colors = [custom_palette[label] for label in sentiment_counts.index]

fig = go.Figure(data=[
    go.Pie(
        labels=sentiment_counts.index,
        values=sentiment_counts.values,
        hole=0.45,                
        marker_colors=pie_colors,
        textinfo='label+percent', 
        textfont=dict(size=16, color='white'),
        hoverinfo='label+percent+value',
        showlegend=False  
    )
])

# --- 4. FINAL LAYOUT ADJUSTMENTS ---
fig.update_layout(
    title_text="Overall Sentiment on Topic: 'AMORE'",
    title_font=dict(size=24, color='black', family="Arial"),
    title_x=0.5,                
    template='plotly_white',
    height=500,                
    margin=dict(t=80, b=40, l=40, r=40)
)

fig.show()